# IBM HR Employee Attrition — Feature Engineering

**Group 6 | IE University MLOps Final Project**

This notebook transforms the raw IBM HR dataset into a clean, model-ready format. The output is consumed directly by all training scripts in Stage 03 and beyond.

**Steps:**
1. Drop useless columns
2. Encode target variable
3. Engineer 4 domain-specific features
4. Encode categorical features
5. Train / test split (80/20, stratified)
6. Scale numeric features (fit on train only — no leakage)
7. Save processed splits

**Note on class imbalance:** Per the proposal, we start with class-weighted learning and threshold tuning. SMOTE is a fallback applied inside training folds in Stage 03 only if class weights prove insufficient. We do NOT apply SMOTE here.

---

## 0. Setup

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120

OUTPUT_DIR = Path('../data/processed')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PLOT_DIR = Path('../data/plots')
PLOT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42

print('Setup complete.')

Setup complete.


## 1. Load Raw Data

In [2]:
df = pd.read_csv('../data/IBM_emp_attrition.csv', sep=';')
print(f'Raw shape: {df.shape}')
df.head(3)

Raw shape: (1470, 35)


,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,...,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,...,1,80,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,2,...,4,80,1,10,3,3,10,7,1,7
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,...,2,80,0,7,3,3,0,0,0,0


## 2. Drop Useless Columns

In [3]:
DROP_COLS = ['EmployeeCount', 'StandardHours', 'Over18', 'EmployeeNumber']
df = df.drop(columns=DROP_COLS)
print(f'Shape after dropping useless columns: {df.shape}')

Shape after dropping useless columns: (1470, 31)


## 3. Encode Target Variable

In [4]:
df['Attrition'] = (df['Attrition'] == 'Yes').astype(int)
print('Attrition encoded: Yes=1, No=0')
print(df['Attrition'].value_counts())

Attrition encoded: Yes=1, No=0
Attrition
0    1233
1     237
Name: count, dtype: int64


## 4. Engineer Domain-Specific Features

Per the proposal, four interpretable features are constructed **before** encoding/scaling so they benefit from the same transformations as raw features. They are validated via ablation in Stage 03.

| Feature | Formula | Rationale |
|---------|---------|----------|
| `PromotionStagnationRatio` | `YearsSinceLastPromotion / max(1, YearsAtCompany)` | Captures career stagnation relative to tenure |
| `WorkloadPayPressure` | `OverTime × MonthlyIncome` | Interaction: being underpaid AND overworked |
| `AverageSatisfaction` | Mean of 4 satisfaction scores | Composite wellbeing index |
| `TenureBucket` | 0–2=0, 3–7=1, 8+=2 | New-hire vulnerability non-linearity |

In [5]:
# PromotionStagnationRatio
df['PromotionStagnationRatio'] = (
    df['YearsSinceLastPromotion'] / df['YearsAtCompany'].clip(lower=1)
)

# WorkloadPayPressure: OverTime (Yes=1) x MonthlyIncome
# High value = high income + overtime (stress despite pay)
# Low value = no overtime OR low income
overtime_binary = (df['OverTime'] == 'Yes').astype(int)
df['WorkloadPayPressure'] = overtime_binary * df['MonthlyIncome']

# AverageSatisfaction: mean of all 4 satisfaction scores
SAT_COLS = ['JobSatisfaction', 'EnvironmentSatisfaction',
            'RelationshipSatisfaction', 'WorkLifeBalance']
df['AverageSatisfaction'] = df[SAT_COLS].mean(axis=1)

# TenureBucket: ordinal — captures non-linear tenure effect
df['TenureBucket'] = pd.cut(
    df['YearsAtCompany'],
    bins=[-1, 2, 7, 100],
    labels=[0, 1, 2]
).astype(int)

print('4 engineered features created:')
eng_cols = ['PromotionStagnationRatio', 'WorkloadPayPressure',
            'AverageSatisfaction', 'TenureBucket']
print(df[eng_cols].describe().round(3))

4 engineered features created:
       PromotionStagnationRatio  WorkloadPayPressure  AverageSatisfaction  \
count                  1470.000             1470.000             1470.000   
mean                      0.290             1853.195                2.731   
std                       0.341             3857.775                0.506   
min                       0.000                0.000                1.000   
25%                       0.000                0.000                2.500   
50%                       0.167                0.000                2.750   
75%                       0.500             2352.500                3.000   
max                       1.000            19859.000                4.000   

       TenureBucket  
count      1470.000  
mean          1.127  
std           0.759  
min           0.000  
25%           1.000  
50%           1.000  
75%           2.000  
max           2.000  


In [6]:
# Quick validation: do engineered features correlate with attrition?
for col in eng_cols:
    corr = df[col].corr(df['Attrition'])
    print(f'  {col:<30}: r = {corr:+.3f}')

  PromotionStagnationRatio      : r = +0.032
  WorkloadPayPressure           : r = +0.075
  AverageSatisfaction           : r = -0.159
  TenureBucket                  : r = -0.188


## 5. Encode Categorical Features

In [7]:
# Binary encoding
df['OverTime'] = (df['OverTime'] == 'Yes').astype(int)
df['Gender']   = (df['Gender']   == 'Male').astype(int)
print('Binary encoded: OverTime (Yes=1), Gender (Male=1)')

# One-hot encoding for all multi-class categoricals
# Per proposal: BusinessTravel, Department, EducationField, JobRole, MaritalStatus
OHE_COLS = ['BusinessTravel', 'Department', 'EducationField', 'JobRole', 'MaritalStatus']
df = pd.get_dummies(df, columns=OHE_COLS, drop_first=False)

# Convert bool columns to int
bool_cols = df.select_dtypes(include='bool').columns
df[bool_cols] = df[bool_cols].astype(int)

print(f'One-hot encoded: {OHE_COLS}')
print(f'Shape after encoding: {df.shape}')

Binary encoded: OverTime (Yes=1), Gender (Male=1)
One-hot encoded: ['BusinessTravel', 'Department', 'EducationField', 'JobRole', 'MaritalStatus']
Shape after encoding: (1470, 54)


## 6. Train / Test Split (80/20, Stratified)

Per the proposal: 80% train, 20% test, stratified to preserve the 16.1% attrition rate in both splits. Cross-validation (5-fold) handles model selection within training data in Stage 03 — no separate validation file needed.

In [8]:
X = df.drop(columns=['Attrition'])
y = df['Attrition']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print(f'Train size : {len(X_train)} rows  |  attrition rate: {y_train.mean()*100:.1f}%')
print(f'Test size  : {len(X_test)} rows   |  attrition rate: {y_test.mean()*100:.1f}%')
print()
print('Both splits preserve the original 16.1% attrition rate — stratification confirmed.')
print('Test set is held out and NOT used until final evaluation in Stage 03.')

Train size : 1176 rows  |  attrition rate: 16.2%
Test size  : 294 rows   |  attrition rate: 16.0%

Both splits preserve the original 16.1% attrition rate — stratification confirmed.
Test set is held out and NOT used until final evaluation in Stage 03.


## 7. Scale Numeric Features

StandardScaler is fitted **only on training data** and applied to both splits. Fitting on test data would constitute data leakage. Scale-sensitive models (Logistic Regression, SVM) require this; tree-based models (Random Forest, XGBoost) are robust to it.

In [9]:
# Numeric columns to scale
# Excludes binary columns and OHE columns (already 0/1)
NUMERIC_COLS = [
    'Age', 'DailyRate', 'DistanceFromHome', 'Education',
    'EnvironmentSatisfaction', 'HourlyRate', 'JobInvolvement',
    'JobLevel', 'JobSatisfaction', 'MonthlyIncome', 'MonthlyRate',
    'NumCompaniesWorked', 'PercentSalaryHike', 'PerformanceRating',
    'RelationshipSatisfaction', 'StockOptionLevel', 'TotalWorkingYears',
    'TrainingTimesLastYear', 'WorkLifeBalance', 'YearsAtCompany',
    'YearsInCurrentRole', 'YearsSinceLastPromotion', 'YearsWithCurrManager',
    # Engineered numeric features
    'PromotionStagnationRatio', 'WorkloadPayPressure',
    'AverageSatisfaction', 'TenureBucket'
]

# Only scale columns that actually exist in X
cols_to_scale = [c for c in NUMERIC_COLS if c in X_train.columns]

scaler = StandardScaler()
X_train[cols_to_scale] = scaler.fit_transform(X_train[cols_to_scale])
X_test[cols_to_scale]  = scaler.transform(X_test[cols_to_scale])

print(f'StandardScaler fitted on training data only.')
print(f'Scaled {len(cols_to_scale)} numeric columns.')
print('No data leakage: test set transformed using training statistics.')

StandardScaler fitted on training data only.
Scaled 27 numeric columns.
No data leakage: test set transformed using training statistics.


## 8. Save Processed Data

In [10]:
train_df = X_train.copy()
train_df['Attrition'] = y_train.values

test_df = X_test.copy()
test_df['Attrition'] = y_test.values

train_df.to_csv(OUTPUT_DIR / 'train.csv', index=False)
test_df.to_csv(OUTPUT_DIR / 'test.csv', index=False)

# Save feature column list (used by all training scripts to guarantee column order)
feature_cols = list(X_train.columns)
with open(OUTPUT_DIR / 'feature_columns.txt', 'w') as f:
    f.write('\n'.join(feature_cols))

# Save scaler for use in deployment (app.py must apply same scaling)
import joblib
joblib.dump(scaler, OUTPUT_DIR / 'scaler.joblib')
joblib.dump(cols_to_scale, OUTPUT_DIR / 'cols_to_scale.joblib')

print('Saved to data/processed/:')
print(f'  train.csv            — {len(train_df)} rows, {train_df["Attrition"].mean()*100:.1f}% attrition')
print(f'  test.csv             — {len(test_df)} rows,  {test_df["Attrition"].mean()*100:.1f}% attrition')
print(f'  feature_columns.txt  — {len(feature_cols)} features')
print(f'  scaler.joblib        — StandardScaler (fitted on train)')
print(f'  cols_to_scale.joblib — list of scaled column names')

Saved to data/processed/:
  train.csv            — 1176 rows, 16.2% attrition
  test.csv             — 294 rows,  16.0% attrition
  feature_columns.txt  — 53 features
  scaler.joblib        — StandardScaler (fitted on train)
  cols_to_scale.joblib — list of scaled column names


## 9. Final Feature Summary

In [11]:
print(f'Total features: {len(feature_cols)}')
print()

raw_numeric = [c for c in feature_cols if c in [
    'Age','DailyRate','DistanceFromHome','Education','EnvironmentSatisfaction',
    'HourlyRate','JobInvolvement','JobLevel','JobSatisfaction','MonthlyIncome',
    'MonthlyRate','NumCompaniesWorked','PercentSalaryHike','PerformanceRating',
    'RelationshipSatisfaction','StockOptionLevel','TotalWorkingYears',
    'TrainingTimesLastYear','WorkLifeBalance','YearsAtCompany','YearsInCurrentRole',
    'YearsSinceLastPromotion','YearsWithCurrManager'
]]
binary_cols  = [c for c in feature_cols if c in ['OverTime', 'Gender']]
ohe_cols_out = [c for c in feature_cols if any(
    c.startswith(p) for p in ['BusinessTravel_','Department_','EducationField_',
                               'JobRole_','MaritalStatus_']
)]
eng_out = ['PromotionStagnationRatio','WorkloadPayPressure',
           'AverageSatisfaction','TenureBucket']

print(f'Raw numeric      ({len(raw_numeric)}): scaled with StandardScaler')
print(f'Binary encoded   ({len(binary_cols)}): OverTime, Gender')
print(f'One-hot encoded  ({len(ohe_cols_out)}): BusinessTravel, Department, EducationField, JobRole, MaritalStatus')
print(f'Engineered       ({len(eng_out)}): {eng_out}')
print()
print('Class imbalance strategy (per proposal):')
print('  Step 1: class_weight="balanced" in all models (Stage 03)')
print('  Step 2: threshold tuning on precision-recall curve (Stage 03)')
print('  Step 3: SMOTE inside CV folds only IF F1 is insufficient (Stage 03)')
print()
print('✅ Feature engineering complete. Next: 03-experiments/attrition_experiments.ipynb')

Total features: 53

Raw numeric      (23): scaled with StandardScaler
Binary encoded   (2): OverTime, Gender
One-hot encoded  (24): BusinessTravel, Department, EducationField, JobRole, MaritalStatus
Engineered       (4): ['PromotionStagnationRatio', 'WorkloadPayPressure', 'AverageSatisfaction', 'TenureBucket']

Class imbalance strategy (per proposal):
  Step 1: class_weight="balanced" in all models (Stage 03)
  Step 2: threshold tuning on precision-recall curve (Stage 03)
  Step 3: SMOTE inside CV folds only IF F1 is insufficient (Stage 03)

✅ Feature engineering complete. Next: 03-experiments/attrition_experiments.ipynb
